# Topic 2 — LangGraph: Practice Notebook

This notebook is the hands-on companion to `notes/02-langgraph.md`. Read that
note first — it explains the *why* and the *math* behind every section below,
with dry runs. This notebook focuses on running the code yourself.

**How this notebook stays offline and deterministic**: every "model" call
uses `GenericFakeChatModel` from `langchain_core` — no API key, no network
call, no Ollama server required.

**Sections**:

1. State Schema & Reducers
2. Nodes & Edges — a small "research assistant" graph
3. Cycles & the ReAct Loop
4. Checkpointers & Persistence
5. Human-in-the-Loop — **EXERCISE**: `human_review_node`
6. Multi-Agent Graphs — **EXERCISE**: a 2-agent supervisor
7. Streaming

Cells marked **EXERCISE** contain a function signature with a docstring
describing exactly what to implement, followed by `# YOUR CODE HERE` and
`pass`. Replace `pass` with your implementation. The `solutions/` copy of
this notebook starts identical to this one — work through it there, and check
`solutions/02-langgraph-code-explanation.md` for the canonical implementation,
full walkthroughs, and dry runs once you're done.


In [ ]:
from typing import TypedDict, Annotated

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

print("Imports OK. GenericFakeChatModel stands in for a real chat model below.")


## 1. State Schema & Reducers

A LangGraph graph is built around one shared **state** object, usually a
`TypedDict`. Every node is a function `state -> partial update`, and each
field of the state has a **reducer** that decides how a node's update is
combined with the existing value. Plain fields (e.g. `topic: str`) use the
default reducer — overwrite. The `messages` field is annotated
`Annotated[list, add_messages]`, whose reducer appends new messages, or
replaces an existing message if the new one shares its `id`.


In [ ]:
m1 = [HumanMessage(content="What is LangGraph?", id="1")]
m2 = [AIMessage(content="LangGraph is a library for building stateful agents.", id="2")]

merged = add_messages(m1, m2)
for message in merged:
    print(type(message).__name__, message.id, "->", message.content)

# A new message with the SAME id as an existing one REPLACES it
m3 = [AIMessage(content="LangGraph models LLM apps as graphs of nodes and edges.", id="2")]
merged_again = add_messages(merged, m3)
print()
for message in merged_again:
    print(type(message).__name__, message.id, "->", message.content)


### Why the reducer matters

Without `add_messages`, the default reducer (overwrite) would mean every
node's `messages` update *replaces* the whole list — only the last node to
touch `messages` would survive. The two graphs below run the same two nodes
(`a` then `b`, each appending one message) with different state schemas.


In [ ]:
class NoReducerState(TypedDict):
    messages: list

class ReducerState(TypedDict):
    messages: Annotated[list, add_messages]

def node_a(state):
    return {"messages": [HumanMessage(content="from node_a")]}

def node_b(state):
    return {"messages": [AIMessage(content="from node_b")]}

no_reducer_graph = StateGraph(NoReducerState)
no_reducer_graph.add_node("a", node_a)
no_reducer_graph.add_node("b", node_b)
no_reducer_graph.add_edge(START, "a")
no_reducer_graph.add_edge("a", "b")
no_reducer_graph.add_edge("b", END)
no_reducer_app = no_reducer_graph.compile()

result = no_reducer_app.invoke({"messages": [HumanMessage(content="initial")]})
print("without add_messages reducer:")
for message in result["messages"]:
    print(" ", type(message).__name__, repr(message.content))

reducer_graph = StateGraph(ReducerState)
reducer_graph.add_node("a", node_a)
reducer_graph.add_node("b", node_b)
reducer_graph.add_edge(START, "a")
reducer_graph.add_edge("a", "b")
reducer_graph.add_edge("b", END)
reducer_app = reducer_graph.compile()

result = reducer_app.invoke({"messages": [HumanMessage(content="initial")]})
print("\nwith add_messages reducer:")
for message in result["messages"]:
    print(" ", type(message).__name__, repr(message.content))


### The state schema for this notebook

`ResearchState` is the schema used from Section 2 onward: `messages` (with
the `add_messages` reducer), plus three plain fields that each hold *one
current value* for the run — `topic`, `search_results`, and `summary`.


In [ ]:
class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    search_results: str
    summary: str


## 2. Nodes & Edges — a Research Assistant Graph

`add_node(name, fn)` registers a node. `add_edge(a, b)` always goes from `a`
to `b`. `add_conditional_edges(a, router_fn, mapping)` runs `router_fn(state)`
after `a` and uses `mapping` to turn its return value into the next node's
name.

The running example is a tiny "research assistant": `extract_topic` reads the
latest human message, `search` looks it up in a small in-memory index, and
then either `summarize` (if something was found) or `no_results` (if not)
produces the final answer.


In [ ]:
SEARCH_INDEX = {
    "langgraph": "LangGraph is a library for building stateful, multi-step LLM applications as graphs of nodes and edges.",
    "checkpointer": "A checkpointer persists the state of a graph after every step, identified by a thread_id.",
    "react": "The ReAct pattern interleaves reasoning (LLM thoughts) with acting (tool calls) in a loop.",
}

def search_index(query: str) -> str:
    query_lower = query.lower()
    for keyword, snippet in SEARCH_INDEX.items():
        if keyword in query_lower:
            return snippet
    return "No results found."

print(search_index("Tell me about langgraph"))
print(search_index("quantum gravity"))


In [ ]:
research_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="LangGraph models LLM applications as graphs of nodes and edges with persistent state."),
]))

def extract_topic(state: ResearchState) -> dict:
    last_message = state["messages"][-1]
    return {"topic": last_message.content}

def search_node(state: ResearchState) -> dict:
    return {"search_results": search_index(state["topic"])}

def route_after_search(state: ResearchState) -> str:
    if state["search_results"] == "No results found.":
        return "no_results"
    return "summarize"

def summarize_node(state: ResearchState) -> dict:
    prompt = f"Summarize this in one sentence: {state['search_results']}"
    response = research_llm.invoke([HumanMessage(content=prompt)])
    return {"summary": response.content, "messages": [response]}

def no_results_node(state: ResearchState) -> dict:
    message = AIMessage(content=f"I couldn't find anything about '{state['topic']}'.")
    return {"summary": message.content, "messages": [message]}


In [ ]:
research_graph = StateGraph(ResearchState)
research_graph.add_node("extract_topic", extract_topic)
research_graph.add_node("search", search_node)
research_graph.add_node("summarize", summarize_node)
research_graph.add_node("no_results", no_results_node)
research_graph.add_edge(START, "extract_topic")
research_graph.add_edge("extract_topic", "search")
research_graph.add_conditional_edges(
    "search",
    route_after_search,
    {"summarize": "summarize", "no_results": "no_results"},
)
research_graph.add_edge("summarize", END)
research_graph.add_edge("no_results", END)
research_app = research_graph.compile()


In [ ]:
result = research_app.invoke({
    "messages": [HumanMessage(content="langgraph")],
    "topic": "", "search_results": "", "summary": "",
})
print("topic:", result["topic"])
print("search_results:", result["search_results"])
print("summary:", result["summary"])


In [ ]:
result = research_app.invoke({
    "messages": [HumanMessage(content="quantum gravity")],
    "topic": "", "search_results": "", "summary": "",
})
print("topic:", result["topic"])
print("search_results:", result["search_results"])
print("summary:", result["summary"])


## 3. Cycles & the ReAct Loop

A linear graph runs each node once. An agent that can call tools needs a
**cycle**: an `agent` node (the LLM) and a `tools` node (runs whatever the LLM
asked for), with a conditional edge after `agent` that goes to `tools` if the
LLM's reply has `tool_calls`, or to `END` if it doesn't — and a plain edge
from `tools` back to `agent`. LangGraph ships this conditional as
`tools_condition` and the dispatcher as `ToolNode`.

`web_search` wraps `search_index` from Section 2 as a `@tool`.


In [ ]:
@tool
def web_search(query: str) -> str:
    """Search the web for a query and return a one-sentence snippet."""
    return search_index(query)

print(web_search.name)
print(web_search.description)
print(web_search.invoke({"query": "Tell me about LangGraph"}))


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

agent_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "LangGraph"}, "id": "call_1"}]),
    AIMessage(content="LangGraph is great for building agents."),
]))

def agent_node(state: AgentState) -> dict:
    return {"messages": [agent_llm.invoke(state["messages"])]}

agent_graph = StateGraph(AgentState)
agent_graph.add_node("agent", agent_node)
agent_graph.add_node("tools", ToolNode([web_search]))
agent_graph.add_edge(START, "agent")
agent_graph.add_conditional_edges("agent", tools_condition)
agent_graph.add_edge("tools", "agent")
agent_app = agent_graph.compile()


In [ ]:
result = agent_app.invoke({"messages": [HumanMessage(content="What is LangGraph?")]})
for message in result["messages"]:
    print(type(message).__name__, "|", repr(message.content), "| tool_calls:", getattr(message, "tool_calls", None))


## 4. Checkpointers & Persistence

Compiling a graph with `checkpointer=...` saves a full snapshot of the state
— and which node runs next — after every step, keyed by a `thread_id`.
`get_state(config)` returns the latest snapshot; `get_state_history(config)`
returns every snapshot ever recorded for that thread, newest first.


In [ ]:
checkpoint_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "LangGraph"}, "id": "call_1"}]),
    AIMessage(content="LangGraph is great for building agents."),
]))

def checkpoint_agent_node(state: AgentState) -> dict:
    return {"messages": [checkpoint_llm.invoke(state["messages"])]}

checkpoint_graph = StateGraph(AgentState)
checkpoint_graph.add_node("agent", checkpoint_agent_node)
checkpoint_graph.add_node("tools", ToolNode([web_search]))
checkpoint_graph.add_edge(START, "agent")
checkpoint_graph.add_conditional_edges("agent", tools_condition)
checkpoint_graph.add_edge("tools", "agent")

checkpointer = InMemorySaver()
checkpoint_app = checkpoint_graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "thread-1"}}
result = checkpoint_app.invoke({"messages": [HumanMessage(content="What is LangGraph?")]}, config=config)
print("number of messages after invoke:", len(result["messages"]))


In [ ]:
snapshot = checkpoint_app.get_state(config)
print("next:", snapshot.next)
print("number of messages:", len(snapshot.values["messages"]))


In [ ]:
print(f"{'idx':>3} | {'next':<14} | len(messages)")
for idx, snapshot in enumerate(checkpoint_app.get_state_history(config)):
    print(f"{idx:>3} | {str(snapshot.next):<14} | {len(snapshot.values['messages'])}")


### Production note — `SqliteSaver` / `PostgresSaver`

`InMemorySaver` keeps every checkpoint in a Python dict — it disappears when
the process exits. `langgraph-checkpoint-sqlite`'s `SqliteSaver` (and the
Postgres equivalent) write the same checkpoint records to a real database, so
a thread survives restarts. The interface — `get_state`, `get_state_history`,
`invoke(..., config={"configurable": {"thread_id": ...}})` — is identical;
only the constructor changes.


In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect(":memory:", check_same_thread=False)
sqlite_checkpointer = SqliteSaver(conn)
sqlite_app = checkpoint_graph.compile(checkpointer=sqlite_checkpointer)
print(type(sqlite_app.checkpointer).__name__)


## 5. Human-in-the-Loop

Calling `interrupt(payload)` inside a node **pauses the whole graph run** at
that point; `payload` comes back to the caller as part of the result (under
the `"__interrupt__"` key), and the run's state is safely checkpointed.
Calling `app.invoke(Command(resume=value), config)` with the same `thread_id`
reloads that checkpoint, re-enters the node, and `interrupt(payload)` *returns*
`value` — the node continues from that line as if no time had passed.

### Exercise — `human_review_node`

Implement `human_review_node` below. It runs *after* `agent` whenever the
agent's latest message has `tool_calls`, and *before* `tools`. It should call
`interrupt(...)` with a payload describing the proposed tool call, then branch
on the resumed decision.


In [ ]:
def human_review_node(state: AgentState) -> dict:
    """
    Pause for human approval before a requested tool call runs.

    `state["messages"][-1]` is the agent's latest AIMessage, which has a
    non-empty `tool_calls` list (that's why this node was reached).

    1. Call `interrupt(...)` with a dict payload describing the proposed
       call, e.g. {"question": "Approve this tool call?",
                    "tool_call": state["messages"][-1].tool_calls[0]}.
       `interrupt` blocks here until the graph is resumed with
       `Command(resume=decision)`, then RETURNS `decision`.
    2. If decision["type"] == "approve", return {} -- no state change,
       so the graph proceeds to the "tools" node.
    3. Otherwise, raise a ValueError describing the rejection.
    """
    # YOUR CODE HERE
    pass


The rest of the graph wiring is given: `route_after_agent` sends the agent's
output to `human_review` if it requested tools, otherwise straight to `END`.


In [ ]:
def route_after_agent(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "human_review"
    return END

hitl_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "LangGraph"}, "id": "call_1"}]),
    AIMessage(content="LangGraph is great for building agents."),
    AIMessage(content="LangGraph is a graph-based framework for building agents."),
]))

def hitl_agent_node(state: AgentState) -> dict:
    return {"messages": [hitl_llm.invoke(state["messages"])]}

hitl_graph = StateGraph(AgentState)
hitl_graph.add_node("agent", hitl_agent_node)
hitl_graph.add_node("human_review", human_review_node)
hitl_graph.add_node("tools", ToolNode([web_search]))
hitl_graph.add_edge(START, "agent")
hitl_graph.add_conditional_edges("agent", route_after_agent, {"human_review": "human_review", END: END})
hitl_graph.add_edge("human_review", "tools")
hitl_graph.add_edge("tools", "agent")

hitl_checkpointer = InMemorySaver()
hitl_app = hitl_graph.compile(checkpointer=hitl_checkpointer)


In [ ]:
hitl_config = {"configurable": {"thread_id": "thread-2"}}
result = hitl_app.invoke({"messages": [HumanMessage(content="What is LangGraph?")]}, config=hitl_config)
print("interrupt:", result.get("__interrupt__"))
print("next:", hitl_app.get_state(hitl_config).next)


In [ ]:
result = hitl_app.invoke(Command(resume={"type": "approve"}), config=hitl_config)
for message in result["messages"]:
    print(type(message).__name__, "|", repr(message.content))


### Time travel

`get_state_history` returns *every* checkpoint for a thread. Calling
`app.invoke(None, config=<an earlier checkpoint's config>)` re-runs the graph
**from that point forward** — everything before it is replayed from the saved
checkpoint, and everything after it runs again from scratch.


In [ ]:
history = list(hitl_app.get_state_history(hitl_config))
print(f"{'idx':>3} | {'next':<16} | len(messages)")
for idx, snapshot in enumerate(history):
    print(f"{idx:>3} | {str(snapshot.next):<16} | {len(snapshot.values['messages'])}")

fork_point = next(s for s in history if s.next == ("agent",))
print("\nforking from checkpoint with next =", fork_point.next, "and", len(fork_point.values["messages"]), "messages")

forked_result = hitl_app.invoke(None, config=fork_point.config)
for message in forked_result["messages"]:
    print(type(message).__name__, "|", repr(message.content))


## 6. Multi-Agent Graphs

The **supervisor pattern**: one router node looks at the request and
dispatches to one of several specialist nodes, each focused on one kind of
task. (The alternative, the **swarm** pattern, has agents hand off directly to
each other instead of going through a central router.)

### Exercise — a 2-agent supervisor

`SupervisorState` adds `next_agent: str` to `messages`. `route_to_specialist`
(given) reads `next_agent` and picks a specialist, defaulting to
`research_specialist` if the supervisor's answer wasn't `"math"`. Implement
the three node functions below.


In [ ]:
class SupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str

def route_to_specialist(state: SupervisorState) -> str:
    if state.get("next_agent") == "math":
        return "math_specialist"
    return "research_specialist"

supervisor_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="math"),
    AIMessage(content="research"),
]))

def supervisor_node(state: SupervisorState) -> dict:
    """
    Classify the latest message by calling `supervisor_llm` with
    `state["messages"]`, and store its (stripped) reply content in
    `next_agent`. Return {"next_agent": ...}.
    """
    # YOUR CODE HERE
    pass

def math_specialist(state: SupervisorState) -> dict:
    """
    Parse the latest human message as "<a> + <b>", compute the sum, and
    return {"messages": [AIMessage(content=f"The answer is {total}.")]}.
    """
    # YOUR CODE HERE
    pass

def research_specialist(state: SupervisorState) -> dict:
    """
    Look up the latest human message with `search_index` and return
    {"messages": [AIMessage(content=snippet)]}.
    """
    # YOUR CODE HERE
    pass


In [ ]:
supervisor_graph = StateGraph(SupervisorState)
supervisor_graph.add_node("supervisor", supervisor_node)
supervisor_graph.add_node("math_specialist", math_specialist)
supervisor_graph.add_node("research_specialist", research_specialist)
supervisor_graph.add_edge(START, "supervisor")
supervisor_graph.add_conditional_edges(
    "supervisor",
    route_to_specialist,
    {"math_specialist": "math_specialist", "research_specialist": "research_specialist"},
)
supervisor_graph.add_edge("math_specialist", END)
supervisor_graph.add_edge("research_specialist", END)
supervisor_app = supervisor_graph.compile()


In [ ]:
result = supervisor_app.invoke({"messages": [HumanMessage(content="12 + 30")], "next_agent": ""})
print("next_agent:", repr(result["next_agent"]))
print("answer:", result["messages"][-1].content)


In [ ]:
result = supervisor_app.invoke({"messages": [HumanMessage(content="langgraph")], "next_agent": ""})
print("next_agent:", repr(result["next_agent"]))
print("answer:", result["messages"][-1].content)


## 7. Streaming

`app.stream(input, stream_mode=...)` yields output incrementally instead of
waiting for the whole run to finish.

```
"updates"  -- one event per node: {node_name: <what that node changed>}
"values"   -- one event per node: the FULL state so far
"messages" -- one event per LLM token, tagged with which node produced it
```


In [ ]:
stream_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "LangGraph"}, "id": "call_1"}]),
    AIMessage(content="LangGraph is great for building agents."),
]))

def stream_agent_node(state: AgentState) -> dict:
    return {"messages": [stream_llm.invoke(state["messages"])]}

stream_graph = StateGraph(AgentState)
stream_graph.add_node("agent", stream_agent_node)
stream_graph.add_node("tools", ToolNode([web_search]))
stream_graph.add_edge(START, "agent")
stream_graph.add_conditional_edges("agent", tools_condition)
stream_graph.add_edge("tools", "agent")
stream_app = stream_graph.compile()

for update in stream_app.stream({"messages": [HumanMessage(content="What is LangGraph?")]}, stream_mode="updates"):
    for node_name, node_update in update.items():
        message = node_update["messages"][0]
        print(node_name, "->", type(message).__name__, repr(message.content))


In [ ]:
stream_llm_2 = GenericFakeChatModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "LangGraph"}, "id": "call_1"}]),
    AIMessage(content="LangGraph is great for building agents."),
]))

def stream_agent_node_2(state: AgentState) -> dict:
    return {"messages": [stream_llm_2.invoke(state["messages"])]}

stream_graph_2 = StateGraph(AgentState)
stream_graph_2.add_node("agent", stream_agent_node_2)
stream_graph_2.add_node("tools", ToolNode([web_search]))
stream_graph_2.add_edge(START, "agent")
stream_graph_2.add_conditional_edges("agent", tools_condition)
stream_graph_2.add_edge("tools", "agent")
stream_app_2 = stream_graph_2.compile()

for state in stream_app_2.stream({"messages": [HumanMessage(content="What is LangGraph?")]}, stream_mode="values"):
    print("len(messages):", len(state["messages"]))


`"messages"` streaming requires the model's token-by-token `_stream` method
to actually run, which only happens for plain-text content. The agent above
opens with an `AIMessage` whose `content` is `""` (it only carries
`tool_calls`), which produces zero chunks and raises an error under
`"messages"` streaming with `GenericFakeChatModel`. So this demo uses a
small single-node graph with plain-text output instead.


In [ ]:
class SimpleState(TypedDict):
    messages: Annotated[list, add_messages]

respond_llm = GenericFakeChatModel(messages=iter([
    AIMessage(content="LangGraph is great for building agents."),
]))

def respond_node(state: SimpleState) -> dict:
    return {"messages": [respond_llm.invoke(state["messages"])]}

respond_graph = StateGraph(SimpleState)
respond_graph.add_node("respond", respond_node)
respond_graph.add_edge(START, "respond")
respond_graph.add_edge("respond", END)
respond_app = respond_graph.compile()

for token, metadata in respond_app.stream({"messages": [HumanMessage(content="What is LangGraph?")]}, stream_mode="messages"):
    print(repr(token.content), "| node:", metadata["langgraph_node"])


## Putting It All Together

```
START -> extract_topic -> search -+-> summarize ---------> END     (Section 2)
                                   +-> no_results --------> END

START -> agent <--tools_condition--> tools                          (Section 3)
            |  no tool_calls
            v
           END

agent --tool_calls--> human_review --> tools --> agent --> ...      (Section 5)
  |  no tool_calls                       ^
  v                                       | interrupt() / Command(resume=...)
 END                                checkpointer (Section 4)

START -> supervisor -+-> math_specialist -----> END                 (Section 6)
                      +-> research_specialist -> END

app.stream(..., stream_mode="values" | "updates" | "messages")      (Section 7)
```

Every piece above is the same handful of ideas recombined: a `TypedDict`
state with reducers (Section 1), nodes and (conditional) edges (Section 2),
a cycle with an exit condition (Section 3), a checkpointer that makes the
cycle resumable (Section 4), `interrupt()`/`Command` for pausing inside that
cycle (Section 5), more than one specialist node sharing a state (Section 6),
and incremental output at three different granularities (Section 7).


## Where to Go Next

Topic 3 (Model Context Protocol) picks up the `web_search` tool used in
Sections 3, 5, and 6: instead of a Python function defined in this notebook,
MCP standardizes how a node's tools can live in a separate process (or
machine) and still be discovered and called the same way —
`tool_calls -> ToolMessage`, now crossing a process boundary.

Topic 4 (AI Agent Patterns & Frameworks) compares this hand-built supervisor
graph against higher-level abstractions — LangGraph's own prebuilt agents,
CrewAI, the Claude Agent SDK — that wrap the same node/edge/checkpoint
primitives in different APIs.
